# UC-ASR on Kaggle GPU

Runs the **full-scale** UC-ASR pipeline with a real LLM (Qwen2.5-3B-Instruct, 4-bit) on a Kaggle T4/P100.

**Before running:**
1. Settings -> Accelerator -> **GPU T4 x2** (or P100).
2. Settings -> **Internet: On** (needed to pip-install, download the dataset, and pull the HF model). Requires a phone-verified Kaggle account. If you cannot enable internet, attach the raw data as a Kaggle Dataset instead (see the offline cell).
3. Push the repo to GitHub, or upload it as a Kaggle Dataset, and set `REPO_*` below.

The permanent SQLite LLM cache means if a 12h session dies you can re-run and resume from cached reasoning traces.


## 1. Get the code


In [ ]:
# Option A: clone from GitHub (set your repo URL)
REPO_URL = 'https://github.com/<you>/UC_ASR_Thesis.git'  # <-- EDIT
import os, subprocess, sys
WORK = '/kaggle/working/UC_ASR_Thesis'
if not os.path.isdir(WORK):
    subprocess.run(['git','clone','--depth','1',REPO_URL,WORK], check=True)
os.chdir(WORK)
print('cwd =', os.getcwd())
print(os.listdir('.'))


### (Alternative) code uploaded as a Kaggle Dataset
If you added the repo as a Kaggle Dataset instead of GitHub, copy it into the writable working dir:


In [ ]:
# import shutil
# SRC = '/kaggle/input/<your-code-dataset-slug>'
# WORK = '/kaggle/working/UC_ASR_Thesis'
# shutil.copytree(SRC, WORK, dirs_exist_ok=True)
# os.chdir(WORK)


## 2. Install dependencies
`bitsandbytes` installs here because Kaggle is Linux + CUDA (unlike the CPU-only dev box).


In [ ]:
!pip -q install -r requirements.txt
!pip -q install bitsandbytes  # CUDA 4-bit quantization (Linux/CUDA only)


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')


## 3. (Offline only) point at an attached raw-data dataset
Skip this if Internet is On. If offline, attach a Kaggle Dataset containing
`Grocery_and_Gourmet_Food.jsonl.gz` and `meta_Grocery_and_Gourmet_Food.jsonl.gz`.
`load_config` auto-detects it; you can also pin the slug explicitly:


In [ ]:
# os.environ['UC_ASR_KAGGLE_DATASET'] = '<your-raw-data-slug>'  # /kaggle/input/<slug>


## 4. Sanity check config (kaggle scale)
The `kaggle` scale uses the real HF backend, K=50, 5-core, N=3, and caps LLM-scored users to fit one session.


In [ ]:
from src.common import load_config, _on_kaggle
cfg = load_config('config.yaml', 'kaggle')
print('on kaggle:', _on_kaggle())
print('backend :', cfg['llm']['backend'], '| model:', cfg['llm']['hf_model'])
print('4bit    :', cfg['llm']['load_in_4bit'], '| device:', cfg['llm']['device'])
print('k_core  :', cfg['preprocessing']['k_core'], '| users:', cfg['preprocessing']['n_users_subset'])
print('top_k   :', cfg['retrievers']['top_k_candidates'], '| N passes:', cfg['uncertainty']['uaro']['n_passes'])
print('root    :', cfg['project']['root'])


## 5. Run the pipeline
Stage by stage so a session timeout doesn't lose completed stages (outputs are idempotent / cached).
To compare Mistral-7B later, edit `scales.kaggle.llm.hf_model` to `mistralai/Mistral-7B-Instruct-v0.2` and re-run.


In [ ]:
!python run_pipeline.py --scale kaggle --stage data


In [ ]:
!python run_pipeline.py --scale kaggle --stage retrieve


In [ ]:
!python run_pipeline.py --scale kaggle --stage rerank   # warms the LLM cache (the slow part)


In [ ]:
!python run_pipeline.py --scale kaggle --stage eval


## 6. Inspect results


In [ ]:
import pandas as pd, pathlib
print(pathlib.Path('results/thesis_main_table.md').read_text())
display(pd.read_csv('results/thesis_main_table.csv'))
print(pathlib.Path('results/statistical_significance.md').read_text())


In [ ]:
from IPython.display import Image, display
for p in sorted(pathlib.Path('results/figures').glob('*.png')):
    print(p.name); display(Image(str(p)))


## 7. Persist outputs
Everything under `/kaggle/working` is saved as the notebook's output when you **Save Version**.
Download `results/`, the parquet splits, and `cache/llm_responses.sqlite` to resume/extend later.


In [ ]:
!du -sh cache/llm_responses.sqlite results data/processed 2>/dev/null || true
